# Notebook 32 — Tool Registry & Sandboxing

Register, validate, and safely execute agent tools.

| Component | Purpose |
|---|---|
| `ToolSpec` | JSON-Schema descriptor |
| `Tool` | validated callable with call tracking |
| `ToolRegistry` | named tool store + async dispatcher |
| `Sandbox` | timeout + output-size enforcement |
| `SandboxedTool` | tool wrapped in a sandbox |
| `PermissionPolicy` | allow/deny by agent and tool pattern |
| `PermissionedRegistry` | registry with policy enforcement |

## 1. Register tools

In [ ]:
from multigen.tools import (
    ToolCall, ToolResult, ToolSpec, Tool, ToolRegistry,
    Sandbox, SandboxedTool, PermissionPolicy, PermissionedRegistry,
)

registry = ToolRegistry()

@registry.register(
    description='Add two numbers',
    parameters={
        'type': 'object',
        'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}},
        'required': ['a', 'b'],
    },
)
async def add(a: float, b: float) -> float:
    return a + b

@registry.register(
    description='Multiply two numbers',
    parameters={
        'type': 'object',
        'properties': {'x': {'type': 'number'}, 'y': {'type': 'number'}},
        'required': ['x', 'y'],
    },
    tags=['math'],
)
async def multiply(x: float, y: float) -> float:
    return x * y

@registry.register(
    description='Convert text to uppercase',
    parameters={
        'type': 'object',
        'properties': {'text': {'type': 'string'}},
        'required': ['text'],
    },
)
async def uppercase(text: str) -> str:
    return text.upper()

print(f'Registry has {len(registry)} tools')

## 2. Call tools

In [ ]:
# Direct call
result = await registry.call(ToolCall('add', {'a': 3, 'b': 4}))
print(f'add(3,4) = {result.output}  latency={result.latency_ms:.1f}ms')

# Convenience method
result2 = await registry.call_by_name('multiply', x=6, y=7)
print(f'multiply(6,7) = {result2.output}')

result3 = await registry.call_by_name('uppercase', text='hello world')
print(f'uppercase = {result3.output}')

## 3. Input validation

In [ ]:
# Missing required argument
result_bad = await registry.call(ToolCall('add', {'a': 5}))  # missing 'b'
print(f'Missing arg error: {result_bad.error}')

# Wrong type
result_type = await registry.call(ToolCall('uppercase', {'text': 123}))  # int not string
print(f'Type error: {result_type.error}')

## 4. Sandbox — timeout and output limits

In [ ]:
import asyncio

# Register a slow tool
registry2 = ToolRegistry()

@registry2.register(
    description='Slow computation',
    parameters={'type':'object','properties':{'delay':{'type':'number'}},'required':['delay']},
    sandbox=Sandbox(timeout_s=0.5, max_retries=1),
)
async def slow_compute(delay: float) -> str:
    await asyncio.sleep(delay)
    return 'done'

# Fast call succeeds
r_fast = await registry2.call(ToolCall('slow_compute', {'delay': 0.1}))
print(f'Fast call: success={r_fast.success}  output={r_fast.output}')

# Slow call times out
r_slow = await registry2.call(ToolCall('slow_compute', {'delay': 2.0}))
print(f'Slow call: success={r_slow.success}  error={r_slow.error}')

## 5. Permission policy

In [ ]:
policy = PermissionPolicy()
policy.allow(agent_pattern='researcher-*', tool_pattern='*')
policy.allow(agent_pattern='admin', tool_pattern='*')
policy.deny(agent_pattern='*', tool_pattern='delete-*')

# Test
test_cases = [
    ('researcher-1', 'add',         True),
    ('researcher-1', 'delete-user', False),
    ('admin',        'delete-user', True),
    ('guest',        'add',         False),
]
for agent, tool_name, expected in test_cases:
    result = policy.check(agent, tool_name)
    status = 'OK' if result == expected else 'FAIL'
    print(f'[{status}] {agent} calling {tool_name}: {result}')

## 6. Permissioned registry

In [ ]:
preg = PermissionedRegistry(policy=policy)

@preg.register(description='Delete a record', parameters={})
async def delete_user(user_id: str = '') -> str:
    return f'deleted {user_id}'

# Researcher can call 'delete-user'? No — policy denies it
call_as = ToolCall('delete_user', {}, caller_id='researcher-1')
r = await preg.call(call_as)
print(f'researcher calling delete_user: {r.error or r.output}')  # should be denied

call_admin = ToolCall('delete_user', {'user_id': '42'}, caller_id='admin')
r2 = await preg.call(call_admin)
print(f'admin calling delete_user: {r2.output}')

## 7. LLM-provider schemas

In [ ]:
openai_schema = registry.to_openai_schema()
print('OpenAI format:')
for s in openai_schema:
    print(f"  {s['name']}: {s['description']}")

anthropic_schema = registry.to_anthropic_schema()
print('\nAnthropic format (input_schema key):')
for s in anthropic_schema:
    print(f"  {s['name']}: has input_schema={bool(s['input_schema'])}")

## 8. Tool call stats

In [ ]:
print('Registry stats:', registry.stats())